# 05 Schema Evolution

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Change a Delta table's schema <b>on purpose</b>: add columns automatically on append (<code>mergeSchema</code>), replace the schema on overwrite (<code>overwriteSchema</code>), evolve during <code>MERGE</code>, and change columns explicitly with <code>ALTER TABLE</code> (add, comment, reorder, rename and drop with column mapping).
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Sources evolve: a CRM adds a <code>loyalty_tier</code> field, an API adds <code>device_type</code>. Schema enforcement (lesson 04) blocks those changes by default. Schema evolution is how you accept them deliberately, in a controlled way. The exam tests the difference between <code>mergeSchema</code> and <code>overwriteSchema</code>, and what each one allows.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Ways to evolve a schema</b><br>
| Method | Allows | Typical use |<br>|---|---|---|<br>| <code>.option("mergeSchema", "true")</code> on <b>append</b> | Adding new columns (and some safe type widenings) | New optional fields from a source |<br>| <code>.option("overwriteSchema", "true")</code> on <b>overwrite</b> | Any schema: drop, rename, change types | Rebuilding a table with a new design |<br>| <code>MERGE WITH SCHEMA EVOLUTION</code> (or the auto-merge setting) | New columns from the source in <code>UPDATE SET *</code> / <code>INSERT *</code> | Upserts from evolving sources |<br>| <code>ALTER TABLE ... ADD COLUMNS</code> | New columns, explicitly | Planned changes, with comments |<br>| <code>ALTER TABLE ... RENAME COLUMN</code> / <code>DROP COLUMN</code> | Rename or drop <b>without rewriting data</b> | Needs <b>column mapping</b> (<code>delta.columnMapping.mode = 'name'</code>) |<br>| Auto Loader <code>schemaEvolutionMode</code> | New columns while streaming files | Ingestion (folder 08) |<br><br>
Every schema change is a commit, so it appears in the history and can be undone with <code>RESTORE</code>.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A delivery app added <code>tip_amount</code> to its order events. The Bronze ingestion uses Auto Loader with schema evolution, so the column appeared automatically. The Silver <code>MERGE</code> used <code>WITH SCHEMA EVOLUTION</code>, so Silver gained the column too, with nulls for old orders. The Gold revenue table, defined with explicit DDL, stayed unchanged until the team decided how tips should be reported, and then added it with <code>ALTER TABLE ... ADD COLUMNS</code>.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates an <code>orders</code> table used throughout the lesson.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
3 orders and the initial schema.
</div>

In [0]:
DB = "workspace.delta_lake"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
from pyspark.sql import functions as F

table = f"{DB}.orders_evo"
spark.createDataFrame(
    [(1, "C1", 120.0), (2, "C2", 80.0), (3, "C1", 45.0)],
    "order_id INT, customer_id STRING, amount DOUBLE",
).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
spark.table(table).printSchema()

## 1. Adding a column on append with `mergeSchema`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Appends rows with a new <code>channel</code> column: first without options (rejected), then with <code>mergeSchema</code>.<br><br>
<b>Why it matters</b><br><code>mergeSchema</code> adds the new column to the table schema. Existing rows get null for it. It doesn't remove columns or change existing types incompatibly.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The first append fails. The second succeeds, the schema now ends with <code>channel</code>, and old rows show null in it.
</div>

In [0]:
new_batch = spark.createDataFrame([(4, "C3", 300.0, "app")], "order_id INT, customer_id STRING, amount DOUBLE, channel STRING")

try:
    new_batch.write.mode("append").saveAsTable(table)
except Exception as e:
    print("without mergeSchema:", getattr(e, "getCondition", lambda: type(e).__name__)())

new_batch.write.mode("append").option("mergeSchema", "true").saveAsTable(table)
spark.table(table).printSchema()
spark.table(table).orderBy("order_id").show()

## 2. Replacing the schema with `overwriteSchema`

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Overwrites the table with a DataFrame that has a different design: <code>amount</code> becomes <code>DECIMAL(10,2)</code>, and <code>customer_id</code> is dropped and replaced by <code>customer</code>.<br><br>
<b>Why it matters</b><br><code>overwriteSchema</code> allows <b>any</b> change, because the data is fully replaced. Without it, an overwrite still has to match the existing schema. It's powerful and easy to misuse: downstream readers see the new schema immediately.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
An overwrite without the option fails. With <code>overwriteSchema</code>, the table has the new columns and types, and the history shows both versions.
</div>

In [0]:
redesigned = spark.table(table).select(
    "order_id",
    F.col("customer_id").alias("customer"),
    F.col("amount").cast("decimal(10,2)").alias("amount"),
    "channel",
)

try:
    redesigned.write.mode("overwrite").saveAsTable(table)
except Exception as e:
    print("overwrite without overwriteSchema:", getattr(e, "getCondition", lambda: type(e).__name__)())

redesigned.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
spark.table(table).printSchema()

## 3. Explicit changes with ALTER TABLE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Adds two columns with comments at a chosen position, changes a column comment, and moves a column.<br><br>
<b>Why it matters</b><br>For planned changes, explicit DDL is clearer than automatic evolution: you choose names, types, comments and positions, and reviewers see the change in code.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A schema with <code>currency</code> right after <code>amount</code> and <code>discount</code> at the end, and the comment on <code>channel</code>.
</div>

In [0]:
spark.sql(f"ALTER TABLE {table} ADD COLUMNS (currency STRING COMMENT 'ISO 4217 code' AFTER amount, discount DECIMAL(10,2) COMMENT 'Absolute discount')")
spark.sql(f"ALTER TABLE {table} ALTER COLUMN channel COMMENT 'Sales channel: app, web, store'")
spark.sql(f"ALTER TABLE {table} ALTER COLUMN channel FIRST")
spark.sql(f"DESCRIBE TABLE {table}").show(truncate=False)

## 4. Rename and drop columns with column mapping

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Enables <b>column mapping</b> by name, then renames <code>customer</code> to <code>customer_id</code> and drops <code>discount</code>, all without rewriting data files.<br><br>
<b>Why it matters</b><br>Parquet files identify columns by name. Column mapping gives each column a stable internal ID, so renames and drops become metadata-only operations. Enabling it upgrades the table protocol, so older readers that don't support it can no longer read the table.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The <code>delta.columnMapping.mode = name</code> property, then a schema with <code>customer_id</code> and no <code>discount</code>, while the data is unchanged.
</div>

In [0]:
spark.sql(f"""
    ALTER TABLE {table} SET TBLPROPERTIES (
        'delta.columnMapping.mode' = 'name',
        'delta.minReaderVersion' = '2',
        'delta.minWriterVersion' = '5'
    )
""")
spark.sql(f"ALTER TABLE {table} RENAME COLUMN customer TO customer_id")
spark.sql(f"ALTER TABLE {table} DROP COLUMN discount")

print(spark.table(table).columns)
spark.table(table).orderBy("order_id").show()

## 5. Schema evolution in MERGE

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Upserts a source that has a new <code>loyalty_tier</code> column into a customers table, with schema evolution enabled for the <code>MERGE</code>.<br><br>
<b>Why it matters</b><br><code>MERGE</code> is how Silver tables are usually maintained (lesson 07). With <code>WITH SCHEMA EVOLUTION</code> (Databricks), new source columns used by <code>UPDATE SET *</code> / <code>INSERT *</code> are added to the target automatically. In open-source Delta and older runtimes, the equivalent is the session setting <code>spark.databricks.delta.schema.autoMerge.enabled</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The customers table gains a <code>loyalty_tier</code> column: the updated and new customers have values, and the untouched one has null.
</div>

In [0]:
customers = f"{DB}.customers_evo"
spark.createDataFrame([("C1", "Amal"), ("C2", "John"), ("C3", "Vivek")], "customer_id STRING, name STRING") \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(customers)
spark.createDataFrame([("C2", "John Smith", "gold"), ("C4", "Sara", "silver")], "customer_id STRING, name STRING, loyalty_tier STRING") \
    .createOrReplaceTempView("customer_updates")

In [0]:
spark.sql(f"""
    MERGE WITH SCHEMA EVOLUTION INTO {customers} t
    USING customer_updates s ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

In [0]:
spark.table(customers).orderBy("customer_id").show()

## 6. Every schema change is in the history

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Shows the history of the orders table.<br><br>
<b>Why it matters</b><br>Schema changes are commits like any other: auditable, and reversible with <code>RESTORE</code> (lesson 08). Time travel to an old version returns the old schema.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Versions for the writes, the <code>ADD COLUMNS</code>, <code>CHANGE COLUMN</code>, <code>SET TBLPROPERTIES</code>, <code>RENAME COLUMN</code> and <code>DROP COLUMNS</code> operations. Version 0 is read back with its original 3 columns.
</div>

In [0]:
spark.sql(f"DESCRIBE HISTORY {table}").select("version", "operation").show(truncate=False)
print("version 0 columns:", spark.read.option("versionAsOf", 0).table(table).columns)

## Under the hood

```
mergeSchema append      -> commit: new metaData (schema + new column) + add files
overwriteSchema         -> commit: new metaData (any schema) + remove all files + add new files
ADD COLUMNS / COMMENT   -> commit: new metaData only (no data rewrite)
RENAME / DROP COLUMN    -> commit: new metaData; with column mapping, files keep physical column IDs
                           (DROP COLUMN leaves the data in the files until a rewrite / REORG)
```

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> adding, renaming, dropping and commenting columns are metadata-only changes.

**Type changes:** changing a column's type (for example `INT` → `STRING`) generally requires rewriting the data with `overwriteSchema`. Databricks also supports **type widening** for some safe changes (such as `INT` → `BIGINT`) as a table feature.

In [0]:
spark.sql(f"DESCRIBE DETAIL {table}").select("minReaderVersion", "minWriterVersion", "properties").show(truncate=False)

In [0]:
# Clean up
for t in [table, customers]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: using <code>overwriteSchema</code> when you only wanted to add a column</b><br>
It replaces all data and can silently drop columns. Use <code>mergeSchema</code> on append, or <code>ALTER TABLE ADD COLUMNS</code>.<br><br>
**⛔ Problem: <code>mergeSchema</code> didn't change a column type**<br>It adds columns (and allows only safe widening). Use <code>overwriteSchema</code> for real type changes.<br><br>
**⛔ Problem: <code>RENAME COLUMN</code> / <code>DROP COLUMN</code> is not supported**<br>Enable column mapping first (<code>delta.columnMapping.mode = 'name'</code>).<br><br>
<b>⛔ Problem: an external tool can't read the table any more</b><br>Column mapping upgraded the protocol. Check that the reader supports it before enabling.<br><br>
<b>⛔ Problem: Gold tables change shape unexpectedly</b><br>Schema evolution was enabled everywhere. Allow evolution in Bronze and Silver, and keep Gold schemas explicit.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between schema enforcement and schema evolution?</b><br>
Enforcement rejects writes that don't match the table schema. Evolution deliberately lets the schema change, for example adding new columns, through options such as <code>mergeSchema</code>, <code>overwriteSchema</code> or <code>ALTER TABLE</code>.<br><br>

<b>🎤 2. What is the difference between <code>mergeSchema</code> and <code>overwriteSchema</code>?</b><br>
<code>mergeSchema</code> is used with appends (and merges) to add new columns to the existing schema, keeping the existing data. <code>overwriteSchema</code> is used with an overwrite to replace the schema entirely, including dropping columns or changing types, because all the data is replaced.<br><br>

<b>🎤 3. How do you rename or drop a column in a Delta table without rewriting the data?</b><br>
Enable column mapping by name (<code>delta.columnMapping.mode = 'name'</code>), then use <code>ALTER TABLE ... RENAME COLUMN</code> or <code>DROP COLUMN</code>. Those become metadata-only changes.<br><br>

<b>🎤 4. How do you handle new columns in a <code>MERGE</code>?</b><br>
On Databricks, with <code>MERGE WITH SCHEMA EVOLUTION</code>, or with the session setting <code>spark.databricks.delta.schema.autoMerge.enabled</code>. New source columns used by <code>UPDATE SET *</code> or <code>INSERT *</code> are then added to the target.<br><br>

<b>🎤 5. Where would you allow automatic schema evolution in a medallion architecture?</b><br>
In Bronze and usually Silver, where capturing new source fields is valuable. Gold tables keep explicit schemas, because dashboards and consumers depend on them.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A daily append to a Delta table now includes a new column, and the team wants the table to include it. Which option should they add to the append?</b><br>
A. <code>.option("overwriteSchema", "true")</code><br>
B. <code>.option("mergeSchema", "true")</code><br>
C. <code>.option("inferSchema", "true")</code><br>
D. <code>.option("ignoreChanges", "true")</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b> <code>overwriteSchema</code> is for overwrites that replace the schema.</details><br><br>

<b>Q2. Which table property must be set before running <code>ALTER TABLE ... RENAME COLUMN</code> on a Delta table?</b><br>
A. <code>delta.enableChangeDataFeed</code><br>
B. <code>delta.columnMapping.mode = 'name'</code><br>
C. <code>delta.autoOptimize.optimizeWrite</code><br>
D. <code>delta.appendOnly</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create <code>products (product_id INT, name STRING, price DOUBLE)</code> with 2 rows</li><li>Append a row with an extra <code>category</code> column using <code>mergeSchema</code>, and show the nulls for the old rows</li><li>Add a <code>currency</code> column with a comment, positioned after <code>price</code></li><li>Enable column mapping and rename <code>name</code> to <code>product_name</code></li><li>Overwrite the table so that <code>price</code> becomes <code>DECIMAL(10,2)</code>, and show the history</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
products = f"{DB}.products_evo"

# 1. Initial table
spark.createDataFrame([(1, "Laptop", 900.0), (2, "Mouse", 20.0)], "product_id INT, name STRING, price DOUBLE") \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(products)

# 2. New column on append
spark.createDataFrame([(3, "Desk", 250.0, "furniture")], "product_id INT, name STRING, price DOUBLE, category STRING") \
    .write.mode("append").option("mergeSchema", "true").saveAsTable(products)
spark.table(products).orderBy("product_id").show()

# 3. Explicit column with a comment and position
spark.sql(f"ALTER TABLE {products} ADD COLUMNS (currency STRING COMMENT 'ISO 4217' AFTER price)")

# 4. Column mapping, then rename
spark.sql(f"ALTER TABLE {products} SET TBLPROPERTIES ('delta.columnMapping.mode' = 'name', 'delta.minReaderVersion' = '2', 'delta.minWriterVersion' = '5')")
spark.sql(f"ALTER TABLE {products} RENAME COLUMN name TO product_name")

# 5. Type change needs a rewrite with overwriteSchema
spark.table(products).withColumn("price", F.col("price").cast("decimal(10,2)")) \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(products)
spark.table(products).printSchema()
spark.sql(f"DESCRIBE HISTORY {products}").select("version", "operation").show()
spark.sql(f"DROP TABLE {products}")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Enforcement blocks schema changes. Evolution allows them on purpose</li><li><code>mergeSchema</code> (append/merge): add columns. <code>overwriteSchema</code> (overwrite): replace the schema</li><li><code>ALTER TABLE ADD COLUMNS</code> for planned, documented changes</li><li>Rename and drop columns cheaply with column mapping (<code>delta.columnMapping.mode = 'name'</code>)</li><li><code>MERGE WITH SCHEMA EVOLUTION</code> (Databricks) or the <code>autoMerge</code> setting evolves during upserts</li><li>Every schema change is a commit: auditable, and reversible with time travel and <code>RESTORE</code></li></ul>
</div>

| Task | Code |
|---|---|
| Add columns on append | `df.write.mode("append").option("mergeSchema", "true")` |
| Replace schema | `df.write.mode("overwrite").option("overwriteSchema", "true")` |
| Add column | `ALTER TABLE t ADD COLUMNS (c STRING COMMENT '...' AFTER x)` |
| Rename / drop | `ALTER TABLE t RENAME COLUMN a TO b`, `DROP COLUMN c` (column mapping) |
| Merge with evolution | `MERGE WITH SCHEMA EVOLUTION INTO t USING s ON ...` |

## Git commit

```
git add 03_delta_lake/05_schema_evolution.ipynb
git commit -m "add 03_05 schema evolution notebook"
```